In [1]:
import os
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

# ClinVar

In [2]:
BENCH_DIR = "/home/dnanexus/data_dir/other_benchmarks"

In [3]:
# Download ClinVar VCF (GRCh38) and its index
!mkdir -p {BENCH_DIR}/clinvar
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz -O {BENCH_DIR}/clinvar/clinvar.vcf.gz
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi -O {BENCH_DIR}/clinvar/clinvar.vcf.gz.tbi

File ‘/home/dnanexus/data_dir/other_benchmarks/clinvar/clinvar.vcf.gz’ already there; not retrieving.
File ‘/home/dnanexus/data_dir/other_benchmarks/clinvar/clinvar.vcf.gz.tbi’ already there; not retrieving.


In [14]:
# 1. Create a dummy dataframe with your example string (added CLNSIG for demonstration)
data = {
    "info_col": [
        ".11:g.917887G>T;CLNVC=single_nucleotide_variant;CLNVCSO=SO:0001483;GENEINFO=LINC02593:100130417;MC=SO:0001627|intron_variant;ORIGIN=0",
        "CLNVC=single_nucleotide_variant;CLNSIG=Likely_benign;MC=SO:0001583|missense_variant;ORIGIN=1",
        "CLNVC=deletion;CLNSIG=Pathogenic;MC=SO:0001587|stop_gained;ORIGIN=1"
    ]
}
df = pl.DataFrame(data)

# 2. Extract the fields using Regex
# We create two new columns by parsing the 'info_col'
df.with_columns(
    # Regex explanation for MC: 
    # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
    variant_region = pl.col("info_col").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    
    # Regex explanation for CLNSIG: 
    # Look for "CLNSIG=", capture everything until the next semicolon
    clinical_significance = pl.col("info_col").str.extract(r"CLNSIG=([^;]+)", 1)
)

info_col,variant_region,clinical_significance
str,str,str
""".11:g.917887G>T;CLNVC=single_n…","""intron_variant""",null
"""CLNVC=single_nucleotide_varian…","""missense_variant""","""Likely_benign"""
"""CLNVC=deletion;CLNSIG=Pathogen…","""stop_gained""","""Pathogenic"""


In [16]:
clinvar = (
    pl.read_csv(f"{BENCH_DIR}/clinvar/clinvar.vcf.gz", comment_prefix="##", separator="\t", ignore_errors=True)
    .with_columns(
        # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
        variant_region = pl.col("INFO").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    
        # Look for "CLNSIG=", capture everything until the next semicolon
        clinical_significance = pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1)
    )
)

clinvar

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,variant_region,clinical_significance
i64,i64,i64,str,str,str,str,str,str,str
1,66926,3385321,"""AG""","""A""",""".""",""".""","""ALLELEID=3544463;CLNDISDB=Huma…","""intron_variant""","""Uncertain_significance"""
1,69134,2205837,"""A""","""G""",""".""",""".""","""ALLELEID=2193183;CLNDISDB=MedG…","""missense_variant""","""Likely_benign"""
1,69241,4562067,"""C""","""T""",""".""",""".""","""ALLELEID=4679177;CLNDISDB=MedG…","""missense_variant""","""Uncertain_significance"""
1,69308,3925305,"""A""","""G""",""".""",""".""","""ALLELEID=4039319;CLNDISDB=MedG…","""missense_variant""","""Uncertain_significance"""
1,69314,3205580,"""T""","""G""",""".""",""".""","""ALLELEID=3374047;CLNDISDB=MedG…","""missense_variant""","""Uncertain_significance"""
…,…,…,…,…,…,…,…,…,…
null,274185,3778023,"""C""","""T""",""".""",""".""","""ALLELEID=3894028;CLNDISDB=MedG…","""intron_variant""","""Likely_benign"""
null,274366,2206666,"""G""","""C""",""".""",""".""","""ALLELEID=2200058;CLNDISDB=MedG…","""missense_variant""","""Uncertain_significance"""
null,275068,2241971,"""T""","""C""",""".""",""".""","""ALLELEID=2226217;CLNDISDB=MedG…","""missense_variant""","""Uncertain_significance"""


# TraitGym

In [20]:
from huggingface_hub import hf_hub_download

# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="mendelian_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# Read the local file
trait_gym_m = pl.read_parquet(file_path)
trait_gym_m

chrom,pos,ref,alt,OMIM,consequence,label,tss_dist,match_group
str,i64,str,str,str,str,bool,i64,str
"""1""",1425822,"""C""","""G""",null,"""PLS""",false,48,"""PLS_4"""
"""1""",1615869,"""C""","""T""",null,"""PLS""",false,35,"""PLS_0"""
"""1""",1659060,"""G""","""A""",null,"""PLS""",false,47,"""PLS_4"""
"""1""",1659114,"""A""","""G""",null,"""PLS""",false,101,"""PLS_5"""
"""1""",2050958,"""T""","""C""",null,"""5_prime_UTR_variant""",false,149,"""5_prime_UTR_variant_7"""
…,…,…,…,…,…,…,…,…
"""X""",155613005,"""C""","""T""",null,"""PLS""",false,52,"""PLS_52"""
"""X""",155719093,"""C""","""A""",null,"""5_prime_UTR_variant""",false,4,"""5_prime_UTR_variant_101"""
"""X""",155881342,"""A""","""C""",null,"""PLS""",false,2,"""PLS_57"""


In [21]:
# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="complex_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# Read the local file
trait_gym_c = pl.read_parquet(file_path)
trait_gym_c

complex_traits_matched_9/test.parquet:   0%|          | 0.00/411k [00:00<?, ?B/s]

chrom,pos,ref,alt,pip,trait,label,maf,ld_score,consequence,tss_dist,match_group
str,i64,str,str,f64,str,bool,f64,f64,str,i64,str
"""1""",867476,"""C""","""T""",0.00156,"""""",false,0.079465,44.053,"""non_coding_transcript_exon_var…",56446,"""non_coding_transcript_exon_var…"
"""1""",868052,"""T""","""C""",0.001791,"""""",false,0.077747,44.057,"""non_coding_transcript_exon_var…",55870,"""non_coding_transcript_exon_var…"
"""1""",868635,"""A""","""G""",0.004349,"""""",false,0.075255,43.639,"""non_coding_transcript_exon_var…",55287,"""non_coding_transcript_exon_var…"
"""1""",870176,"""T""","""A""",0.0,"""""",false,0.084371,37.271,"""non_coding_transcript_exon_var…",53746,"""non_coding_transcript_exon_var…"
"""1""",1052930,"""A""","""G""",0.001467,"""""",false,0.058385,46.907,"""non_coding_transcript_exon_var…",18823,"""non_coding_transcript_exon_var…"
…,…,…,…,…,…,…,…,…,…,…,…
"""22""",50368376,"""T""","""C""",0.0,"""""",false,0.19181,86.507,"""dELS""",3695,"""dELS_204"""
"""22""",50571623,"""C""","""T""",0.0,"""""",false,0.061159,23.9,"""dELS""",6291,"""dELS_202"""
"""22""",50671289,"""G""","""A""",0.0,"""""",false,0.036223,12.733,"""pELS_flank""",3125,"""pELS_flank_26"""


In [23]:
trait_gym_c['consequence'].value_counts(sort=True)

consequence,count
str,u64
"""dELS""",3140
"""intron_variant""",1900
"""dELS_flank""",1670
"""intergenic_variant""",1030
"""pELS""",1010
…,…
"""CTCF-only_flank""",80
"""DNase-H3K4me3_flank""",50
"""DNase-H3K4me3""",40


# GeneticGym

In [22]:
!mkdir -p {BENCH_DIR}/genetics_gym
!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/dd-with-combined-controls.tsv.bgz -O {BENCH_DIR}/genetics_gym/dd-with-combined-controls.tsv.bgz
!unzip -f {BENCH_DIR}/genetics_gym/dd-with-combined-controls.tsv.bgz

!wget -nc https://storage.googleapis.com/genetics-gym/evaluation_tables/schema_evaluation_table.tsv.bgz -O {BENCH_DIR}/genetics_gym/schema_evaluation_table.tsv.bgz

File ‘/home/dnanexus/data_dir/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz’ already there; not retrieving.
Archive:  /home/dnanexus/data_dir/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /home/dnanexus/data_dir/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz or
        /home/dnanexus/data_dir/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz.zip, and cannot find /home/dnanexus/data_dir/other_benchmarks/genetics_gym/dd-with-combined-controls.tsv.bgz.ZIP, period.
File ‘/home/dnanexus/data_dir/other_benchmarks/genetics_gym/schema_evaluation_table.tsv.bgz’ already there; not retrieving.


# ProteinGym

In [ ]:
VERSION = "v1.3"
FILENAME = "DMS_ProteinGym_substitutions.zip"
TARGET_DIR = f"{BENCH_DIR}/protein_gym"
TARGET_FILE = f"{TARGET_DIR}/{FILENAME}"

# 1. Create Directory
!mkdir -p {TARGET_DIR}

# 2. Download only if missing
if not os.path.exists(TARGET_FILE):
    print("File not found. Downloading...")
    !curl -o {TARGET_FILE} https://marks.hms.harvard.edu/proteingym/ProteinGym_{VERSION}/{FILENAME}
else:
    print("File already exists. Skipping download.")

# 3. Unzip with "Yes to All" (-o)
# We use -o to overwrite if it exists, avoiding the interactive prompt
print("Unzipping...")
!unzip -o {TARGET_FILE} -d {TARGET_DIR}